In [2]:
import pandas as pd 
import numpy as np 
import json
import os, re 

ztf_id_pattern = re.compile(r'ZTF\d+')


column_names = ['ZTFID', 'type', 'redshift', 'hosted']

img_jsons_path = '/Users/xinyuesheng/Documents/astro_projects/data/image_sets_jsons'
mags_path = '/Users/xinyuesheng/Documents/astro_projects/data/mag_sets_v4'
hosts_path = '/Users/xinyuesheng/Documents/astro_projects/data/host_info_r5_ext'

tde_z_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/tde_with_z.json'
tde_z_2024_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/tde_with_z_2024.json'
slsn_z_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/slsn_with_z.json'
slsn_z_2024_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/slsn_with_z_2024.json'
downloaded_z_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/untouched_2025/bts_new.csv'

tde_z_dict = json.load(open(tde_z_path, 'r'))
tde_z_2024_dict = json.load(open(tde_z_2024_path, 'r'))
slsn_z_dict = json.load(open(slsn_z_path, 'r'))
slsn_z_2024_dict = json.load(open(slsn_z_2024_path, 'r'))


downloaded_df = pd.read_csv(downloaded_z_path)
downloaded_df['redshift'] = downloaded_df['redshift'].replace('-', np.nan)
downloaded_df.dropna(inplace=True)

downloaded_z_dict = dict(zip(downloaded_df['ZTFID'], downloaded_df['redshift']))

tde_z_dict = {**tde_z_dict, **tde_z_2024_dict}
slsn_z_dict = {**slsn_z_dict, **slsn_z_2024_dict}

z_dict = {**tde_z_dict, **slsn_z_dict, **downloaded_z_dict}


img_jsons_list = os.listdir(img_jsons_path)
mags_list = os.listdir(mags_path)
hosts_list = os.listdir(hosts_path)

img_jsons_list = [obj for obj in img_jsons_list if ztf_id_pattern.match(obj)]
mags_list = [obj for obj in mags_list if ztf_id_pattern.match(obj)]
hosts_list = [obj for obj in hosts_list if ztf_id_pattern.match(obj)]

print(len(img_jsons_list), len(mags_list), len(hosts_list))




rows = []

for obj in img_jsons_list:

    try: 
        obj_row = {}
        obj_json = os.path.join(img_jsons_path, obj, 'image_meta.json')
        with open(obj_json, 'r') as f:
            obj_json_dict = json.load(f)
        obj_type = obj_json_dict['label']

        if obj in z_dict:
            redshift = z_dict[obj]
        else:
            obj_mag = os.path.join(mags_path, obj+'.json')
            with open(obj_mag, 'r') as f:
                obj_mag_dict = json.load(f)
            if  obj_mag_dict != None: 
                if 'TNS' in obj_mag_dict and 'z' in obj_mag_dict['TNS']:
                    obj_z = obj_mag_dict['TNS']['z']
                    redshift = obj_z
                else:
                    redshift = None
            else:
                redshift = None

        obj_row['ZTFID'] = obj
        obj_row['redshift'] = redshift
        obj_row['hosted'] = True if obj+'.csv' in hosts_list else False
        obj_row['type'] = obj_type

        rows.append(obj_row)
    except:
        print(obj)
        continue
    

print(len(rows))

df = pd.DataFrame(rows, columns=column_names)

df.to_csv('/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/ztf_train_valid_set.csv', index=True)






5824 5828 5532
ZTF18abnonwz
ZTF18aaqpjja
5822


In [3]:
import pandas as pd
from config import RAW_LABEL_DICT

df = pd.read_csv('/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/ztf_train_valid_set.csv')
df['label'] = df['type'].map(RAW_LABEL_DICT['classify'])
df.to_csv('/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/info/ztf_train_valid_set.csv', index=False)
# df.type.value_counts()
